# 🎭 Funciones como ciudadanos de primera clase
*(versión explicada paso a paso)*

---

## La gran idea que cambia todo

En Python, una función **es un valor más**, igual que un número o una cadena. Eso significa que con una función puedes hacer todo lo que harías con cualquier valor:

1. **Asignarla a una variable**.
2. **Pasarla como argumento** a otra función.
3. **Devolverla como resultado** de otra función.

Esto se conoce como *funciones como ciudadanos de primera clase* y es **una característica fundamental** del lenguaje. Abre la puerta a un estilo de programación muy elegante y a herramientas que verás en el **Tema 6** (`map`, `filter`, `sorted` con `key`...).

## 1) Funciones asignadas a variables

Una función, **sin paréntesis**, es solo el objeto función. Con paréntesis, **se invoca**.

In [ ]:
def saludar(nombre: str) -> None:
    print(f'¡Hola, {nombre}!')

# Asignamos la función a otra variable
saluda = saludar     # ojo: SIN paréntesis

# Ahora 'saluda' es OTRO NOMBRE para la misma función
saluda('María')
saludar('Luis')

# Comprobamos que apuntan al mismo objeto
print(saluda is saludar)

> 💡 **¿Para qué sirve esto en la práctica?** Imagina que tienes una función con un nombre muy largo y descriptivo (`calcular_distancia_euclidea_entre_puntos`) que vas a usar muchas veces. Puedes hacer `d = calcular_distancia_euclidea_entre_puntos` y usar `d(p1, p2)` en lugar del nombre largo.

## 2) Funciones pasadas como argumento

Aquí está donde la cosa se vuelve realmente útil. Vamos a escribir una función `aplicar_dos_veces(f, x)` que recibe **otra función** `f` y un valor `x`, y devuelve `f(f(x))`:

In [ ]:
def aplicar_dos_veces(f, x):
    """Devuelve f(f(x))."""
    return f(f(x))

def cuadrado(x):
    return x * x

print(f'cuadrado(3) = {cuadrado(3)}')                                # 9
print(f'aplicar_dos_veces(cuadrado, 3) = {aplicar_dos_veces(cuadrado, 3)}')  # (3²)² = 81

**Esto es muy potente.** La función `aplicar_dos_veces` es **agnóstica**: no le importa qué hace `f`. Podemos pasarle cualquier función:

In [ ]:
import math

print(aplicar_dos_veces(math.sqrt, 16))   # √(√16) = 2.0
print(aplicar_dos_veces(abs, -7))         # abs(abs(-7)) = 7

## 3) Una aplicación clásica: integración numérica

Una función que recibe **otra función como argumento** se llama *función de orden superior*. Un ejemplo matemático muy útil es la integración numérica por la regla del trapecio: queremos una función capaz de **integrar cualquier función real entre dos límites**, sin tener que reescribir la regla del trapecio cada vez.

In [ ]:
def integral_trapecio(f, a: float, b: float, n: int) -> float:
    """
    Aproxima la integral de f entre a y b con la regla del trapecio compuesta
    (n subintervalos).
    """
    h = (b - a) / n
    suma = (f(a) + f(b)) / 2
    for i in range(1, n):
        suma = suma + f(a + i * h)
    return suma * h

# Usémosla con varias funciones
def f1(x):
    return x ** 2

print(f'∫₀¹ x² dx ≈ {integral_trapecio(f1, 0, 1, 1000):.6f}  (exacto 1/3)')
print(f'∫₀^π sin(x) dx ≈ {integral_trapecio(math.sin, 0, math.pi, 1000):.6f}  (exacto 2)')

**Una sola función `integral_trapecio` nos integra cualquier cosa.** Ese es el poder de las funciones de orden superior. Compara con cómo habrías tenido que escribir un script entero para cada caso en el Tema 4.

## 4) Expresiones `lambda`: funciones anónimas

A veces necesitas una función **diminuta** que solo vas a usar **una vez**. En el ejemplo anterior, definir `def f1(x): return x ** 2` solo para pasarla a `integral_trapecio` es algo excesivo.

Para estos casos, Python permite definir funciones **anónimas en una sola línea** con `lambda`:

```python
lambda parametros: expresión
```

que es **equivalente** a:

```python
def nombre(parametros):
    return expresión
```

Pero sin el `def`, sin el `return` y sin el nombre. Ideal para pasársela a una función de orden superior:

In [ ]:
# Estas dos llamadas son equivalentes:
print(integral_trapecio(f1, 0, 1, 1000))                # con función nombrada
print(integral_trapecio(lambda x: x**2, 0, 1, 1000))    # con lambda

# Y podemos integrar muchas cosas distintas sin tener que definirlas con def:
print(f'∫₀¹ (x² + 1) dx ≈ {integral_trapecio(lambda x: x**2 + 1, 0, 1, 1000):.6f}')
print(f'∫₀¹ eˣ dx ≈ {integral_trapecio(lambda x: math.exp(x), 0, 1, 1000):.6f}')
print(f'    (exacto e - 1 ≈ {math.e - 1:.6f})')

> ⚠️ **No abuses de `lambda`**. Son fantásticas cuando la función es:
>
> * **Pequeña** (una expresión).
> * **De usar y tirar** (no la reutilizas).
> * Necesaria **donde se usa** (no merece la pena dar a luz un `def` con nombre).
>
> Si tu lambda crece más allá de una expresión simple, **vuelve a `def`**. La legibilidad es más importante que la brevedad.

## 5) Funciones que devuelven funciones (closures)

También se pueden **devolver funciones**. Mira esta "fábrica" de funciones potencia:

In [ ]:
def crear_potencia(exponente):
    """Devuelve una NUEVA función que eleva su argumento al exponente dado."""
    def potencia(base):
        return base ** exponente
    return potencia

# Construimos varias funciones especializadas
elevar_al_cuadrado = crear_potencia(2)
elevar_al_cubo = crear_potencia(3)
raiz_cuadrada = crear_potencia(0.5)

print(elevar_al_cuadrado(5))   # 25
print(elevar_al_cubo(2))       # 8
print(raiz_cuadrada(9))        # 3.0

> 🎯 **Magia**: cada función `potencia` devuelta **recuerda el valor de `exponente`** que se usó al crearla. Esto se llama *closure* y es uno de los conceptos más bonitos de la programación funcional.

## 🎯 Conceptos del Tema 5 que has practicado

* ✅ **Asignación** de funciones a variables (sin paréntesis).
* ✅ **Funciones de orden superior**: que reciben o devuelven otras funciones.
* ✅ **Expresiones `lambda`** para funciones anónimas de una sola línea.
* ✅ **Closures**: funciones que recuerdan el contexto en el que fueron creadas.

## 🚀 Anticipo del Tema 6

En el Tema 6 verás que muchas funciones útiles de Python son **funciones de orden superior**:

* `map(funcion, iterable)`: aplica `funcion` a cada elemento.
* `filter(predicado, iterable)`: filtra los elementos según `predicado`.
* `sorted(iterable, key=funcion)`: ordena según el criterio que `funcion` calcula.

Todas reciben **otra función como argumento**, justo lo que has aprendido aquí. Y muchas veces, esa función se pasa **como `lambda`** en línea. Por ejemplo (esto YA es Tema 6, no lo escribas todavía):

```python
sorted([(3, 'a'), (1, 'b'), (2, 'c')], key=lambda par: par[0])
```

Te resultará familiar. Ya lo entiendes 😊.